[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/40_linear_regression.ipynb)

# 🟡 Medium: Linear Regression

Implement **linear regression** using three different approaches — all in pure PyTorch.

Given data `X` of shape `(N, D)` and targets `y` of shape `(N,)`, find weight `w` of shape `(D,)` and bias `b` (scalar) such that:

$$\hat{y} = Xw + b$$

### Signature
```python
class LinearRegression:
    def closed_form(self, X: Tensor, y: Tensor) -> tuple[Tensor, Tensor]: ...
    def gradient_descent(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
    def nn_linear(self, X: Tensor, y: Tensor, lr=0.01, steps=1000) -> tuple[Tensor, Tensor]: ...
```

All methods return `(w, b)` where `w` has shape `(D,)` and `b` has shape `()`.

### Method 1 — Closed-Form (Normal Equation)
Augment X with a ones column, then solve:

$$\theta = (X_{aug}^T X_{aug})^{-1} X_{aug}^T y$$

Or use `torch.linalg.lstsq` / `torch.linalg.solve`.

### Method 2 — Gradient Descent from Scratch
Initialize `w` and `b` to zeros. Repeat for `steps` iterations:
```
pred = X @ w + b
error = pred - y
grad_w = (2/N) * X^T @ error
grad_b = (2/N) * error.sum()
w -= lr * grad_w
b -= lr * grad_b
```

### Method 3 — PyTorch nn.Linear
Create `nn.Linear(D, 1)`, use `nn.MSELoss` and an optimizer (e.g., `torch.optim.SGD`).
After training, extract `w` and `b` from the layer.

### Rules
- All inputs and outputs must be **PyTorch tensors**
- Do **NOT** use numpy or sklearn
- `closed_form` must not use iterative optimization
- `gradient_descent` must manually compute gradients (no `autograd`)
- `nn_linear` should use `torch.nn.Linear` and `loss.backward()`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 2.8 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn

In [12]:
# ✏️ YOUR IMPLEMENTATION HERE


class LinearRegression:
    def closed_form(self, X: torch.Tensor, y: torch.Tensor):
        """Normal equation: w = (X^T X)^{-1} X^T y"""

        X = torch.cat( ( X, torch.ones(X.size(0),1) ) , dim=1 )
        w =torch.linalg.inv( ( X.transpose(0,1) @ X ) ) @ X.transpose(0,1) @ y

        b = w[-1]
        return (w[:-1], b)

    def gradient_descent(self, X: torch.Tensor, y: torch.Tensor,
                         lr: float = 0.01, steps: int = 1000):

        w = torch.zeros(3, X.size(0))
        b = torch.zeros(1,1)
        # steps = 10
        for _ in range(steps):
          pred = X @ w + b
          error = pred - y
          grad_w = (2/X.size(0)) * X.transpose(0,1) @ error
          grad_b = (2/X.size(0)) * error.sum()
          w -= lr * grad_w
          b -= lr * grad_b

          return (w,b)
        """Manual gradient descent loop"""
        # pass  # Return (w, b)

    def nn_linear(self, X: torch.Tensor, y: torch.Tensor,
                  lr: float = 0.01, steps: int = 1000):
        """Train nn.Linear with autograd"""
        pass  # Return (w, b)

In [4]:
print(torch.zeros(1,1))

tensor([[0.]])


In [17]:
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")
# print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

Grad descent: w=tensor([[-1.4447e-02,  6.2259e-03, -1.0289e-02, -1.4873e-02, -4.2701e-03,
         -6.9612e-03, -2.9470e-03, -1.2906e-02, -1.1343e-02, -1.4708e-02,
         -2.3462e-03, -5.8107e-03, -1.0469e-02, -8.7083e-03, -2.9916e-03,
         -2.0456e-02, -4.6432e-03, -1.2754e-02, -3.4929e-03, -8.6265e-03,
         -9.1281e-04, -1.1798e-02, -1.7600e-02, -1.0677e-02, -5.3651e-03,
         -7.3435e-03,  1.6307e-03, -8.3712e-03, -1.0229e-02, -5.2618e-03,
          1.4007e-03, -6.2472e-04,  5.2525e-03, -6.7262e-03, -1.3639e-02,
         -1.0407e-02, -5.6032e-03, -9.4346e-03, -4.1918e-03, -9.5257e-03,
         -6.9544e-03, -8.9497e-03, -1.2404e-02, -1.4642e-02, -3.5265e-03,
         -7.5002e-03,  4.2836e-03, -1.5079e-02, -2.2497e-03, -1.3726e-02,
         -3.9850e-03, -1.5407e-02,  5.3161e-03, -1.6490e-02, -8.7899e-03,
         -2.0280e-03,  2.2671e-03, -1.2492e-02, -6.3145e-03, -9.7933e-03,
         -5.3625e-03, -4.2792e-03,  1.1666e-03, -3.8226e-03, -1.9749e-02,
         -9.1407e-03, 

In [18]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([[-1.4447e-02,  6.2259e-03, -1.0289e-02, -1.4873e-02, -4.2701e-03,
         -6.9612e-03, -2.9470e-03, -1.2906e-02, -1.1343e-02, -1.4708e-02,
         -2.3462e-03, -5.8107e-03, -1.0469e-02, -8.7083e-03, -2.9916e-03,
         -2.0456e-02, -4.6432e-03, -1.2754e-02, -3.4929e-03, -8.6265e-03,
         -9.1281e-04, -1.1798e-02, -1.7600e-02, -1.0677e-02, -5.3651e-03,
         -7.3435e-03,  1.6307e-03, -8.3712e-03, -1.0229e-02, -5.2618e-03,
          1.4007e-03, -6.2472e-04,  5.2525e-03, -6.7262e-03, -1.3639e-02,
         -1.0407e-02, -5.6032e-03, -9.4346e-03, -4.1918e-03, -9.5257e-03,
         -6.9544e-03, -8.9497e-03, -1.2404e-02, -1.4642e-02, -3.5265e-03,
         -7.5002e-03,  4.2836e-03, -1.5079e-02, -2.2497e-03, -1.3726e-02,
         -3.9850e-03, -1.5407e-02,  5.3161e-03, -1.6490e-02, -8.7899e-03,
         -2.0280e-03,  2.2671e-03, -1.2492e-02, -6.3145e-03, -9.7933e-03,
         -5.3625e-03, -4.2792e-03,

TypeError: cannot unpack non-iterable NoneType object

In [16]:
# 🧪 Debug
torch.manual_seed(42)
X = torch.randn(100, 3)
true_w = torch.tensor([2.0, -1.0, 0.5])
y = X @ true_w + 3.0

model = LinearRegression()

w_cf, b_cf = model.closed_form(X, y)
print(f"Closed-form:  w={w_cf}, b={b_cf.item():.4f}")

w_gd, b_gd = model.gradient_descent(X, y, lr=0.05, steps=2000)
print(f"Grad descent: w={w_gd}, b={b_gd.item():.4f}")

w_nn, b_nn = model.nn_linear(X, y, lr=0.05, steps=2000)
print(f"nn.Linear:    w={w_nn}, b={b_nn.item():.4f}")

print(f"\nTrue:         w={true_w}, b=3.0")

Closed-form:  w=tensor([ 2.0000, -1.0000,  0.5000]), b=3.0000
Grad descent: w=tensor([[-1.4447e-02,  6.2259e-03, -1.0289e-02, -1.4873e-02, -4.2701e-03,
         -6.9612e-03, -2.9470e-03, -1.2906e-02, -1.1343e-02, -1.4708e-02,
         -2.3462e-03, -5.8107e-03, -1.0469e-02, -8.7083e-03, -2.9916e-03,
         -2.0456e-02, -4.6432e-03, -1.2754e-02, -3.4929e-03, -8.6265e-03,
         -9.1281e-04, -1.1798e-02, -1.7600e-02, -1.0677e-02, -5.3651e-03,
         -7.3435e-03,  1.6307e-03, -8.3712e-03, -1.0229e-02, -5.2618e-03,
          1.4007e-03, -6.2472e-04,  5.2525e-03, -6.7262e-03, -1.3639e-02,
         -1.0407e-02, -5.6032e-03, -9.4346e-03, -4.1918e-03, -9.5257e-03,
         -6.9544e-03, -8.9497e-03, -1.2404e-02, -1.4642e-02, -3.5265e-03,
         -7.5002e-03,  4.2836e-03, -1.5079e-02, -2.2497e-03, -1.3726e-02,
         -3.9850e-03, -1.5407e-02,  5.3161e-03, -1.6490e-02, -8.7899e-03,
         -2.0280e-03,  2.2671e-03, -1.2492e-02, -6.3145e-03, -9.7933e-03,
         -5.3625e-03, -4.2792e-03,

TypeError: cannot unpack non-iterable NoneType object

In [15]:
# ✅ SUBMIT
from torch_judge import check
check("linear_regression")


🧪 Testing: Linear Regression (Medium)
──────────────────────────────────────────────────
  ✅ [1/6] Closed-form returns correct shapes (2.7ms)
  ✅ [2/6] Closed-form finds correct weights (2.3ms)
  💥 [3/6] Gradient descent converges
     RuntimeError: The size of tensor a (100) must match the size of tensor b (3) at non-singleton dimension 1
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<test:Gradient descent converges>", line 10, in <module>
RuntimeError: The size of tensor a (100) must match the size of tensor b (3) at non-singleton dimension 1
  💥 [4/6] nn.Linear approach works
     TypeError: cannot unpack non-iterable NoneType object
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<test:nn.Linear approach works>", line 9, in <module>
TypeError: cannot unpack non-iterable NoneType object
  💥 [5/6] All three methods agree
     RuntimeError: mat1 and mat2 shapes cannot be multiplied (200x2 and 3x200)
     